In [1]:
from maxent_insect_dir import config
from maxent_insect_dir.raw_data_utilis import safe_read_file
import numpy as np 
import pandas as pd
import rasterio
import geopandas as gpd
from shapely.geometry import Point

import matplotlib.pyplot as plt
import os

In [3]:
files = sorted(os.listdir(config.config_worldclim['yearly_avg']['1970_2020']), key = lambda x : int(x.split('_')[-1].split('.')[0]))

In [ ]:
files[0]

In [ ]:
files[0].split('_')[-2].upper()

In [ ]:
point_layer = config.config_raw['insect']['EDD']['Brown_Marmorated_Stink_bug']

# Load the point layer using geopandas (assuming the point layer is in a shapefile)
points = safe_read_file(point_layer)

In [ ]:
points.crs

In [ ]:
for idx, file in enumerate(files):
    raster_path = os.path.join(config.config_worldclim['yearly_avg']['1970_2020'], file)
    var_name = 'WORLDCLIM_YEARLY_AVG_BIO'+str(idx + 1)
    # Load the raster file using rasterio
    with rasterio.open(raster_path) as src:
        print(f'insect data crs : {points.crs} | bioclim data crs : {src.crs}')
        assert points.crs == src.crs, 'mismatch in the crs co-ord of two file'
        
        # Sample raster values at point coordinates
        points[var_name] = [
            list(src.sample([(geom.x, geom.y)]))[0][0] if geom.is_valid else None
            for geom in points.geometry
        ]
        
        

In [9]:
df_ = safe_read_file("/data/Maxent/data/interim/WorldClim/Root_Weevil_GBIF_intermediatory.gpkg")

In [ ]:
_ = df_.hist(figsize=(25,25), grid=False)

In [ ]:
df_.describe()

In [ ]:

# Check for -inf values in the DataFrame
has_neg_inf = (df_['WORLDCLIM_YEARLY_AVG_BIO1'] == -np.inf).values.any()
has_neg_inf

In [ ]:
df_[(df_['WORLDCLIM_YEARLY_AVG_BIO2'] < -1e10) | (df_['WORLDCLIM_YEARLY_AVG_BIO2'] > 1e10)]

In [16]:
df_ = df_.drop(df_[(df_['WORLDCLIM_YEARLY_AVG_BIO2'] < -1e10) | (df_['WORLDCLIM_YEARLY_AVG_BIO2'] > 1e10)].index)

# Pop tmax, tmin and precp dat from  Worldclim monthly avg

In [2]:
point_layer = config.config_interim_WorldClim['insect']['Brown_Marmorated_Stink_bug']

# Load the point layer using geopandas (assuming the point layer is in a shapefile)
insect_data = safe_read_file(point_layer)

In [ ]:
insect_data.head(2)

In [ ]:
insect_data['month']

In [ ]:
config.config_worldclim['monthly_avg']

In [6]:
# Group the insect point data by year
grouped = insect_data.groupby('year')

In [ ]:
config.config_worldclim['monthly_avg'].keys()

In [ ]:
def get_nearest_time_range(year, data):
    nearest_range = None
    smallest_diff = float('inf')
    
    for range_key in data.keys():
        start_year, end_year = map(int, range_key.split('-'))
        
        # If the year is within the range, return it immediately
        if start_year <= year <= end_year:
            return data[range_key], range_key, year
        
        # Calculate the smallest difference between the year and the range boundaries
        diff = min(abs(year - start_year), abs(year - end_year))
        if diff < smallest_diff:
            smallest_diff = diff
            nearest_range = range_key
            
            # keep track of the boundary of interest
            if abs(year - start_year) <= abs(year - end_year):
                boundary = start_year
            else:
                boundary = end_year
            
    return data[nearest_range], nearest_range, boundary

# Example usage:
year = 1998
result = get_nearest_time_range(year, config.config_worldclim['monthly_avg'])
result

In [ ]:
results  = []

for year, group in grouped:
    # Count the number of samples for this year
    sample_count = group['objectid'].count()
    print(f'Processing {sample_count} samples for the year {year}...\n')
    year_dict,year_range,boundary = get_nearest_time_range(year, config.config_worldclim['monthly_avg'])
    
    if boundary != year:
        year = boundary
        print(f'updating the {year} to {boundary} as we stepped out of bounds')
        
    
    # load all the tmax, tmin, and prec in a dict for that particular year
    month_dict = {
        'tmax' : {month : f'wc2.1_10m_tmax_{year}-{month:02d}.tif' for month in range(1, 13)},
        'tmin' : {month : f'wc2.1_10m_tmin_{year}-{month:02d}.tif' for month in range(1, 13)},
        'prec' : {month : f'wc2.1_10m_prec_{year}-{month:02d}.tif' for month in range(1, 13)}
    }
    
    final_dict_path = {
        'tmax' : {month : os.path.join(year_dict['tmax'], month_dict['tmax'][month]) for month in range(1, 13)},
        'tmin' : {month : os.path.join(year_dict['tmin'], month_dict['tmin'][month]) for month in range(1, 13)},
        'prec' : {month : os.path.join(year_dict['prec'], month_dict['prec'][month]) for month in range(1, 13)}
    }
    
    data = {
        'tmax' : {month : rasterio.open(final_dict_path['tmax'][month]) for month in range(1, 13)},
        'tmin' : {month : rasterio.open(final_dict_path['tmin'][month]) for month in range(1, 13)},
        'prec' : {month : rasterio.open(final_dict_path['prec'][month]) for month in range(1, 13)}
    }
    print(data)
    
    # Iterate over each insect point in the group
    for idx, row in group.iterrows():
        insect_location = [(row.geometry.x, row.geometry.y)]
        #month = str(row['month']) if row['month'] >= 10 else '0'+str(row['month'])
        month = row['month']
        
        # get these value from the respective raster
        current_tmax_raster = data['tmax'][month]
        current_tmin_raster = data['tmin'][month]
        current_prec_raster = data['prec'][month]
        
        if idx == 0:
            assert group.crs == current_tmax_raster.crs == current_tmin_raster.crs == current_prec_raster.crs, 'check crs system'
        
        tmax = list(current_tmax_raster.sample(insect_location))[0][0]
        tmin = list(current_tmin_raster.sample(insect_location))[0][0]
        prec = list(current_prec_raster.sample(insect_location))[0][0]
        
        print(tmax, tmin, prec)
        
        results.append({
                    'insect_id': row['objectid'],
                    'WORLDCLIM_MONTHLY_AVG_tmax' : tmax,
                    'WORLDCLIM_MONTHLY_AVG_tmin' : tmin,
                    'WORLDCLIM_MONTHLY_AVG_prec' : prec
                })
        
    if year == 2004:
        import sys; sys.exit()

In [ ]:
data['tmax'][1]

In [ ]:
year = 2012
file_names = {f'wc2.1_10m_tmax_{year}-{month:02d}.tif' for month in range(1, 13)}
file_names

# Let's wrap the code above in a function to inject in the pipeline.

In [9]:

def get_nearest_time_range(year, data):
    nearest_range = None
    smallest_diff = float('inf')
    
    for range_key in data.keys():
        start_year, end_year = map(int, range_key.split('-'))
        
        # If the year is within the range, return it immediately
        if start_year <= year <= end_year:
            return data[range_key], range_key, year
        
        # Calculate the smallest difference between the year and the range boundaries
        diff = min(abs(year - start_year), abs(year - end_year))
        if diff < smallest_diff:
            smallest_diff = diff
            nearest_range = range_key
            
            # keep track of the boundary of interest
            if abs(year - start_year) <= abs(year - end_year):
                boundary = start_year
            else:
                boundary = end_year
            
    return data[nearest_range], nearest_range, boundary


def worldclimbioclim_monthlyyavg_insect_intersection(climate_data_path, insect_crop_data, id_column):
    """
    This function intersects climate data with insect point data on a monthly and yearly basis.
    For each insect point, it retrieves the temperature maximum (tmax), temperature minimum (tmin),
    and precipitation (prec) values for the corresponding month from the climate data.
    
    Args:
        climate_data_path (dict): Dictionary with year ranges as keys and paths to climate data as values.
        insect_crop_data (pd.DataFrame): DataFrame containing insect data, including coordinates, month, and year.
        id_column (str): Column name for unique identifiers of insect points.
    
    Returns:
        pd.DataFrame: The original insect data merged with monthly average climate data (tmax, tmin, prec).
    """
    
    results = []

    # Group the insect point data by year for processing efficiency
    grouped = insect_crop_data.groupby('year')
    
    # Loop over each year group
    for year, group in grouped:
        # Count the number of samples for this year
        sample_count = group[id_column].count()
        print(f'Processing {sample_count} samples for the year {year}...\n')

        # Get the nearest available climate data for the year
        year_dict, year_range, boundary = get_nearest_time_range(year, climate_data_path)
        
        if boundary != year:
            year = boundary
            print(f'updating the {year} to {boundary} as we stepped out of bounds')
        
        # Construct file paths for tmax, tmin, and prec climate data for each month
        month_dict = {
            'tmax': {month: f'wc2.1_10m_tmax_{year}-{month:02d}.tif' for month in range(1, 13)},
            'tmin': {month: f'wc2.1_10m_tmin_{year}-{month:02d}.tif' for month in range(1, 13)},
            'prec': {month: f'wc2.1_10m_prec_{year}-{month:02d}.tif' for month in range(1, 13)}
        }

        # Build final paths using the year-specific climate data directories
        final_dict_path = {
            'tmax': {month: os.path.join(year_dict['tmax'], month_dict['tmax'][month]) for month in range(1, 13)},
            'tmin': {month: os.path.join(year_dict['tmin'], month_dict['tmin'][month]) for month in range(1, 13)},
            'prec': {month: os.path.join(year_dict['prec'], month_dict['prec'][month]) for month in range(1, 13)}
        }
        
        # Open all the raster files for the year and month
        data = {
            'tmax': {month: rasterio.open(final_dict_path['tmax'][month]) for month in range(1, 13)},
            'tmin': {month: rasterio.open(final_dict_path['tmin'][month]) for month in range(1, 13)},
            'prec': {month: rasterio.open(final_dict_path['prec'][month]) for month in range(1, 13)}
        }
    
        # Iterate over each insect point in the group
        for idx, row in group.iterrows():
            insect_location = [(row.geometry.x, row.geometry.y)]
            month = row['month']
            
            # Retrieve the current month's climate data rasters
            current_tmax_raster = data['tmax'][month]
            current_tmin_raster = data['tmin'][month]
            current_prec_raster = data['prec'][month]
            
            # Ensure that all the rasters and the insect data have the same CRS (coordinate reference system)
            if idx == 0:
                assert group.crs == current_tmax_raster.crs == current_tmin_raster.crs == current_prec_raster.crs, 'CRS mismatch detected. Ensure all data uses the same coordinate reference system.'
            
            # Sample the raster data at the insect location
            tmax = list(current_tmax_raster.sample(insect_location))[0][0]
            tmin = list(current_tmin_raster.sample(insect_location))[0][0]
            prec = list(current_prec_raster.sample(insect_location))[0][0]
            
            #print(f'Insect {row[id_column]} - Tmax: {tmax}, Tmin: {tmin}, Prec: {prec}')
            
            # Append the results for this insect point
            results.append({
                'insect_id': row[id_column],
                'WORLDCLIM_MONTHLY_AVG_tmax': tmax,
                'WORLDCLIM_MONTHLY_AVG_tmin': tmin,
                'WORLDCLIM_MONTHLY_AVG_prec': prec
            })
    
    # Convert results into a DataFrame
    result_df = pd.DataFrame(results)
    
    # Merge the climate data back into the original insect point DataFrame
    insect_crop_data = pd.merge(insect_crop_data, result_df, how='left', left_on=id_column, right_on='insect_id')
    
    return insect_crop_data

In [ ]:
insect_data_with_monthly = worldclimbioclim_monthlyyavg_insect_intersection(
    climate_data_path = config.config_worldclim['monthly_avg'],
    insect_crop_data = insect_data,
    id_column =  'objectid'
)

In [ ]:
insect_data_with_monthly.hist()